In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import joblib

import warnings
warnings.filterwarnings("ignore")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv(
    "/content/drive/MyDrive/Internship project/05_processed_text_dataset.csv"
)

print(df.shape)

df.head()

(1206, 11)


,Message_ID,Person,Chat_Name,Timestamp,Sender,Message,Risk_Label,Risk_Category,Confidence,Processed_Message,Tokens
0,1,ARAVIND MENON,Chat with Vishnu,2026-01-05 06:15:00,Vishnu,"Da, report kandille?",Normal,NaN,High,da report kandille,"['da', 'report', 'kandille']"
1,2,ARAVIND MENON,Chat with Vishnu,2026-01-05 06:17:00,You,Kandu. Ellam okay alle?,Normal,NaN,High,kandu ellam okay alle,"['kandu', 'ellam', 'okay', 'alle']"
2,3,ARAVIND MENON,Chat with Vishnu,2026-01-05 06:18:00,Vishnu,Mostly okay. Pakshe aa last item kurachu stran...,Suspicious,Coded Language,Medium,mostly okay pakshe aa last item kurachu strang...,"['mostly', 'okay', 'pakshe', 'aa', 'last', 'it..."
3,4,ARAVIND MENON,Chat with Vishnu,2026-01-05 06:20:00,You,Entha issue?,Normal,NaN,High,entha issue,"['entha', 'issue']"
4,5,ARAVIND MENON,Chat with Vishnu,2026-01-05 06:21:00,Vishnu,Numbers match cheyyunnilla. Randu places il di...,Suspicious,Coded Language,Medium,number match cheyyunnilla randu place il diffe...,"['number', 'match', 'cheyyunnilla', 'randu', '..."


In [ ]:
df["Risk_Label"].value_counts()

,count
Risk_Label,
Normal,905
Suspicious,231
High Risk,70


In [ ]:
X = df["Processed_Message"]

y = df["Risk_Label"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_train_cleaned = X_train.fillna('')
X_test_cleaned = X_test.fillna('')

X_train_tfidf = vectorizer.fit_transform(X_train_cleaned)

X_test_tfidf = vectorizer.transform(X_test_cleaned)

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model.fit(
    X_train_tfidf,
    y_train
)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [ ]:
y_pred = model.predict(
    X_test_tfidf
)

In [ ]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:", round(accuracy,4))

Accuracy: 0.6983


In [ ]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

   High Risk       0.28      0.36      0.31        14
      Normal       0.84      0.81      0.82       182
  Suspicious       0.34      0.35      0.34        46

    accuracy                           0.70       242
   macro avg       0.48      0.51      0.49       242
weighted avg       0.71      0.70      0.70       242



In [ ]:
import pandas as pd

cm_df = pd.DataFrame(
    cm,
    index=model.classes_,
    columns=model.classes_
)

cm_df

,High Risk,Normal,Suspicious
High Risk,5,5,4
Normal,7,148,27
Suspicious,6,24,16


In [ ]:
results = pd.DataFrame({
    "Message": X_test.values,
    "Actual_Label": y_test.values,
    "Predicted_Label": y_pred
})

results.to_csv(
    "/content/drive/MyDrive/Internship project/model_predictions.csv",
    index=False
)

print(results.head(20))

                                   Message Actual_Label Predicted_Label
0                                    sheri       Normal          Normal
1                        hotel staff aavum       Normal      Suspicious
2                saturday match marakkalle       Normal          Normal
3                          entha vishesham       Normal          Normal
4                     ayyoo college memory       Normal          Normal
5   dream 03 01 26 05 47 pm medium omitted       Normal          Normal
6                                      NaN       Normal          Normal
7                    ath aarkum kodukkalle       Normal      Suspicious
8                          reach aayal msg       Normal          Normal
9                                      NaN       Normal          Normal
10                                 enthina       Normal          Normal
11                 evening aa side varunno       Normal          Normal
12                       pwoli player aanu       Normal         

In [ ]:
feature_names = vectorizer.get_feature_names_out()

for i, label in enumerate(model.classes_):
    print(f"\nTop words for {label}:")

    top = np.argsort(model.coef_[i])[-15:]

    for index in reversed(top):
        print(feature_names[index])


Top words for High Risk:
അയക
കണ
result
പറയ
unknown
delete
വര
kurachu
strange
explain
ഇന
safe
മത
seriously
maybe

Top words for Normal:
okay
evde
sheri
poda
free
ok
entha
aanu
probably
inn
ippo
sure
enthina
enthada
true

Top words for Suspicious:
place
evdenn
enth
message deleted
deleted
kitti
parayam
ivide
എത
oru
careful
finally
ശര
il
kandu


In [ ]:
joblib.dump(
    model,
    "/content/drive/MyDrive/Internship project/message_classifier.pkl"
)

joblib.dump(
    vectorizer,
    "/content/drive/MyDrive/Internship project/tfidf_vectorizer.pkl"
)

print("Model saved successfully!")

Model saved successfully!


In [ ]:
print("Training Set:", len(X_train))
print("Testing Set :", len(X_test))

print("\nClass Distribution:")
print(df["Risk_Label"].value_counts())

Training Set: 964
Testing Set : 242

Class Distribution:
Risk_Label
Normal        905
Suspicious    231
High Risk      70
Name: count, dtype: int64
